# Customer Churn Prediction Pipeline

End-to-end churn analysis of bank customer data, in four stages:

1. **Data quality first**: duplicates and missing values are ruled out *before* any analysis, so dirty data cannot silently distort every statistic downstream
2. **EDA on a stratified sample**: EDA compute stays bounded even if a much bigger batch arrives in the future; key findings: **age** and **balance** drive churn, especially around the median balance
3. **ML on the full dataset**: preprocessing, a stratified train/test split, and a Delta table partitioned by `Geography` → `is_train`
4. **Model comparison in MLflow**: four model families trained, every run distinctly named in the experiment; the best run is selected as the final model (**Gradient Boosting**)

## Setup & configuration

Single source of truth: every import, path, schema role, and flow knob lives here as a **global**, and the rest of the notebook only reads them — nothing is re-imported or redefined downstream.

* **Paths** — `RAW_DIR` (input `churn.csv`), `PARTITIONED_DIR` (Delta output for the ML branch), `GRAPHS_DIR` (EDA figures), `ML_EXPERIMENT` (MLflow runs)
* **Schema roles** — `ID_COLS` (identifiers, never features), `NUMERICAL_COLS`, `CATEGORICAL_COLS`, `LABEL`
* **Flow knobs** — `SPLIT_SEED` (one seed for the EDA sample *and* the ML split, so both are reproducible), `TARGET_SAMPLE_SIZE` (EDA cap), `TRAIN_FRACTION`

In [0]:
# --- Imports: the ONLY import block in the notebook (spark is built-in) ---
import os
import numpy as np
import matplotlib.pyplot as plt
from matplotlib import ticker
from datetime import date

from pyspark.sql import Window
from pyspark.sql import functions as F
from pyspark.ml.feature import StringIndexer, VectorAssembler, RobustScaler
from pyspark.ml.stat import Correlation
from pyspark.ml.classification import (
    LogisticRegression, NaiveBayes, RandomForestClassifier, GBTClassifier,
)
from pyspark.ml.evaluation import (
    BinaryClassificationEvaluator, MulticlassClassificationEvaluator,
)
import mlflow
from mlflow import MlflowClient

# --- Path layout ---
WORK_PATH = os.getcwd()
VOLUME_PATH = "/Volumes/workspace/default/customer_data"
RAW_DIR = f"{VOLUME_PATH}/raw"
PARTITIONED_DIR = f"{VOLUME_PATH}/partitioned"
GRAPHS_DIR = f"{WORK_PATH}/graphs"
ML_EXPERIMENT = f"{WORK_PATH}/customer_churn_experiments"
CHURN_CSV = f"{RAW_DIR}/churn.csv"

# --- Schema roles (referenced everywhere instead of re-listing columns) ---
ID_COLS = ["RowNumber", "CustomerId", "Surname"]  # identifiers: never features
NUMERICAL_COLS = ["CreditScore", "Age", "Balance", "EstimatedSalary", "NumOfProducts", "Tenure"]
CATEGORICAL_COLS = ["Geography", "Gender"]
LABEL = "Exited"  # churn target (1 = churned)

# --- Flow knobs ---
SPLIT_SEED = 69             # one seed for the EDA sample AND the ML train/test split
TARGET_SAMPLE_SIZE = 10000  # EDA branch cap: bounds compute if a bigger batch arrives
TRAIN_FRACTION = 0.8        # ML branch: top 80% of each class goes to train

# Create the volume directories if they don't already exist
# (mkdirs also creates intermediate/parent directories as needed)
for _dir in (RAW_DIR, PARTITIONED_DIR):
    dbutils.fs.mkdirs(_dir)

print("Directories ensured:", RAW_DIR, "|", PARTITIONED_DIR, "|", GRAPHS_DIR)

## Load raw data

Read `churn.csv` from the raw volume (header + schema inference, permissive mode), then check the row count, schema, and a first sample of records.

In [0]:
churn_csv = (
    spark.read
    .option("header", True)
    .option("inferSchema", True)
    .option("mode", "PERMISSIVE")
    .csv(CHURN_CSV)
)

print("Rows:", churn_csv.count())
churn_csv.printSchema()
display(churn_csv.limit(10))


## Data quality checks

**Why this comes first:** a duplicate or a missing value would silently distort every statistic and every model trained downstream — so the data is proven clean *before* any analysis touches it. Only clean data proceeds to EDA.

* Summary statistics for the numerical columns
* Duplicate row count
* Null/NaN counts per column, unpivoted into a clean vertical table

In [0]:
churn_csv.select(NUMERICAL_COLS).summary().display()

In [0]:
num_duplicates = churn_csv.count() - churn_csv.distinct().count()
print(f"There are {num_duplicates} duplicates in this dataset.")

In [0]:
float_cols = {c for c, dtype in churn_csv.dtypes if dtype in ("float", "double")}
cols = churn_csv.columns

# Compute counts horizontally first
agg_exprs = [
    F.count(
        F.when(F.col(c).isNull() | (F.isnan(c) if c in float_cols else F.lit(False)), c)
    ).alias(c)
    for c in cols
]

# Unpivot into a clean vertical table
stack_expr = ", ".join([f"'{c}', `{c}`" for c in cols])

churn_csv.select(agg_exprs) \
    .select(F.expr(f"stack({len(cols)}, {stack_expr}) as (column_name, null_count)")) \
    .show(len(cols), truncate=False)

## Distributions & stratified EDA sample

EDA deliberately runs on a **stratified random sample** (`TARGET_SAMPLE_SIZE` rows, same seed as the ML split) that preserves the churn/retain proportions. Today the batch is 10k rows so the sample is effectively the whole dataset but if a much bigger batch arrives tomorrow, EDA compute stays capped while the ML branch still always trains on the **full** data.

1. Build the sample, then histograms of the numerical features
2. First churn-vs-retain aggregates: churners are ~7 years older (44.8 vs 37.4) and carry ~18k more balance (91.1k vs 72.7k): **age** and **balance** emerge as the leading churn drivers

In [0]:
# TARGET_SAMPLE_SIZE / SPLIT_SEED come from the setup cell — no local redefinitions.

total_count = churn_csv.count()
counts_per_class = churn_csv.groupBy("Exited").count().collect()

sample_ratio = min(1.0, TARGET_SAMPLE_SIZE / total_count)
fractions = {row["Exited"]: sample_ratio for row in counts_per_class}

# Stratified sample using native expressions (rand + comparison) instead of
# sampleBy, which compiles to a Scala UDF filter that Photon cannot accelerate.
rand_col = "__sample_rand"
keep_row = F.lit(False)
for stratum, fraction in fractions.items():
    keep_row = keep_row | ((F.col("Exited") == stratum) & (F.col(rand_col) < fraction))

some_churn = (
    churn_csv
    .withColumn(rand_col, F.rand(seed=SPLIT_SEED))
    .where(keep_row)
    .drop(rand_col)
)

In [0]:
# Collect only the numerical columns to the driver as Row objects, then plot each in the grid
numerical_rows = some_churn.select(NUMERICAL_COLS).collect()

ncols = 2
nrows = (len(NUMERICAL_COLS) + ncols - 1) // ncols

fig, axes = plt.subplots(nrows, ncols, figsize=(10, 3.5 * nrows))
axes = axes.flatten()

for i, col in enumerate(NUMERICAL_COLS):
    values = [r[col] for r in numerical_rows]
    axes[i].hist(values, bins=30, color="#2b5c8f", alpha=0.7)
    axes[i].set_title(col)
    axes[i].grid(axis="y", linestyle="--", alpha=0.3)

# Hide any unused grid slots
for j in range(len(NUMERICAL_COLS), len(axes)):
    axes[j].set_visible(False)

plt.tight_layout()
plt.show()

In [0]:
some_churn.groupBy("Exited").agg(
    F.round(F.mean("Age"), 1).alias("avgAge"),
    F.round(F.mean("IsActiveMember") * 100, 1).alias("PctActiveMembers"),
    F.round(F.mean("HasCrCard") * 100, 1).alias("PctCrCardHolders"),
    F.round(F.mean("NumOfProducts"), 2).alias("avgNumOfProducts"),
    F.round(F.mean("Balance"), 2).alias("avgBalance"),
    F.round(F.mean("EstimatedSalary"), 2).alias("avgEstimatedSalary"),
    F.round(F.mean("Tenure"), 1).alias("avgTenure")
).show()

some_churn.groupBy("Geography", "Gender").agg(
    F.count("*").alias("TotalCustomers"),
    F.round(F.mean("Balance"),1).alias("avgBalance"),
    F.round(F.mean("CreditScore")).alias("avgScore").cast('int'),
    F.sum("Exited").alias("ChurnedCount")
).sort(F.desc('avgBalance')).show()


## Feature correlation

Pearson correlation heatmap over all encoded features (string columns indexed to numeric), with the ID columns excluded. Purpose: spot redundant predictors *before* modeling — and confirm no feature is so correlated with the label `Exited` that it would trivially encode the answer (a leakage check).

In [0]:

# 1. Exclude ID / identifier columns (ID_COLS from the setup cell)
candidate_cols = [c for c in churn_csv.columns if c not in ID_COLS]

# 2. Encode categorical string columns (Geography, Gender) to numeric for correlation
df_encoded = churn_csv
feature_cols = []

for c, dtype in churn_csv.select(candidate_cols).dtypes:
    if dtype == "string":
        indexed_col = f"{c}_idx"
        indexer = StringIndexer(inputCol=c, outputCol=indexed_col)
        df_encoded = indexer.fit(df_encoded).transform(df_encoded)
        feature_cols.append(indexed_col)
    else:
        feature_cols.append(c)

# 3. Assemble features into a vector and compute Pearson correlation in Spark
assembler = VectorAssembler(inputCols=feature_cols, outputCol="corr_features")
vector_df = assembler.transform(df_encoded).select("corr_features")

corr_matrix = Correlation.corr(vector_df, "corr_features", method="pearson").collect()[0][0].toArray()

# 4. Clean display labels (strip '_idx')
display_labels = [c.replace("_idx", "") for c in feature_cols]
n = len(display_labels)

# 5. Plot Heatmap using pure Matplotlib
fig, ax = plt.subplots(figsize=(10, 8))
im = ax.imshow(corr_matrix, cmap="coolwarm", vmin=-1, vmax=1)

# Set tick labels
ax.set_xticks(np.arange(n))
ax.set_yticks(np.arange(n))
ax.set_xticklabels(display_labels, rotation=45, ha="right")
ax.set_yticklabels(display_labels)

# Add correlation coefficients inside cells
for i in range(n):
    for j in range(n):
        val = corr_matrix[i, j]
        text_color = "white" if abs(val) > 0.5 else "black"
        ax.text(j, i, f"{val:.2f}", ha="center", va="center", color=text_color, fontsize=8)

fig.colorbar(im, ax=ax, fraction=0.046, pad=0.04, label="Pearson Correlation")
ax.set_title("Feature Correlation Heatmap", fontsize=13, pad=12)

plt.tight_layout()
plt.show()

In [0]:
# 1. Bucket ages < 30 into '20' (18-30), >= 80 into '80' (80+), and the rest in 10-year intervals
age_bucket_df = (
    some_churn
    .withColumn(
        "age_bin_start",
        F.when(F.col("Age") < 30, 20)
         .when(F.col("Age") >= 80, 80)
         .otherwise((F.floor(F.col("Age") / 10) * 10).cast("int"))
    )
    .groupBy("age_bin_start")
    .agg(
        F.count("*").alias("total"),
        F.sum(F.when(F.col("Exited") == 1, 1).otherwise(0)).alias("churned_count"),
        F.sum(F.when(F.col("Exited") == 0, 1).otherwise(0)).alias("retained_count"),
    )
    .withColumn("churn_pct", (F.col("churned_count") / F.col("total")) * 100)
    .withColumn("retained_pct", (F.col("retained_count") / F.col("total")) * 100)
    .orderBy("age_bin_start")
)

# 2. Collect as standard Python Row objects
rows = age_bucket_df.collect()

# Format edge labels as '18-30' and '80+'
def format_age_label(bin_start):
    if bin_start == 20:
        return "18-30"
    elif bin_start == 80:
        return "80+"
    return f"{bin_start}-{bin_start + 10}"

x_labels = [format_age_label(r["age_bin_start"]) for r in rows]
churn_pct = [r["churn_pct"] for r in rows]
retained_pct = [r["retained_pct"] for r in rows]
x_pos = range(len(x_labels))

# 3. 100% Stacked Bar Chart with Matplotlib
fig, ax = plt.subplots(figsize=(10, 4.5))

ax.bar(
    x_pos,
    churn_pct,
    label="Churned (1)",
    color="red",
    alpha=0.5,
    width=0.65
)
ax.bar(
    x_pos,
    retained_pct,
    bottom=churn_pct,
    label="Retained (0)",
    color="green",
    alpha=0.5,
    width=0.65
)

# Formatting
ax.set_xticks(x_pos)
ax.set_xticklabels(x_labels, rotation=45)
ax.set_xlabel("Age Group")
ax.set_ylabel("Percentage (%)")
ax.set_ylim(0, 100)
ax.set_title("Retention vs. Churn Rate by Age Group (100% Stacked)")
ax.legend(loc="upper left")
ax.grid(axis="y", linestyle="--", alpha=0.3)

plt.tight_layout()
# Save the figure directly into the graphs volume (direct write works on this compute)
plt.savefig(f"{GRAPHS_DIR}/churn_by_age_group.png")
plt.show()

## Churn drivers
### Visual analysis

The aggregates above pointed at **age** and **balance** — especially around the **median balance (~97k)** — as the main churn drivers. The figures below (saved to `GRAPHS_DIR`) confirm and sharpen those findings:

* Retention vs churn rate by **age group** (100% stacked bars) -> churn increases with age but decrease after ~50.
* Rolling churn rate by **balance**, split by member activity (windowed SQL) -> around the median balance, active and inactive members diverge sharply but overall active members churn 15% less on average
* Customer volume vs churn rate by number of products -> 3+ products churn almost universally
* Churn rate by geography and by gender (SQL aggregates) -> Germany (~32%) churns at roughly twice the rate of France/Spain

In [0]:
some_churn.createOrReplaceTempView("churn")

In [0]:
sql_query = """
    SELECT 
        Balance,
        IsActiveMember,
        ROUND(
            AVG(Exited) OVER (
                PARTITION BY IsActiveMember
                ORDER BY Balance
                ROWS BETWEEN 500 PRECEDING AND 500 FOLLOWING
            ) * 100.0, 
            2
        ) AS rolling_churn_rate
    FROM churn
    WHERE Balance > 0
    ORDER BY Balance
"""

# 1. Collect as a Python list of Row objects (no pandas)
rows = spark.sql(sql_query).collect()

# 2. Plot using list comprehensions to filter and extract columns
fig, ax = plt.subplots(figsize=(10, 5))

for active_status, label, color in [
    (0, "Inactive Member (0)", "#d95f02"),
    (1, "Active Member (1)", "#2b5c8f")
]:
    balances = [r["Balance"] * 1e-3 for r in rows if r["IsActiveMember"] == active_status]
    rates = [r["rolling_churn_rate"] for r in rows if r["IsActiveMember"] == active_status]
    
    ax.plot(
        balances, 
        rates, 
        label=label, 
        color=color, 
        linewidth=2
    )

ax.set_ylim(10, 40)
ax.xaxis.set_major_formatter(ticker.FuncFormatter(lambda x, pos: f"{x:.0f}k"))
ax.set_xlabel("Balance")
ax.set_ylabel("Churn Probability (%)")
ax.set_title("Continuous Churn Rate by Balance (Split by Member Activity)")
ax.legend(loc="upper right")
ax.grid(True, linestyle="--", alpha=0.3)

plt.tight_layout()
plt.savefig(f"{GRAPHS_DIR}/churn_rate_by_balance.png")
plt.show()

In [0]:
# Aggregate in Spark, collect Row objects for plotting (notebook convention:
# Spark computes, matplotlib draws plain Python lists — no pandas)
prod_rows = (
    some_churn.groupBy("NumOfProducts")
    .agg(
        F.count("*").alias("Total"),
        (F.mean("Exited") * 100).alias("ChurnRate")
    )
    .orderBy("NumOfProducts")
    .collect()
)

fig, ax1 = plt.subplots(figsize=(8, 4.5))

# Bar chart: Total Customers
color = "#2b5c8f"
ax1.set_xlabel("Number of Products")
ax1.set_ylabel("Total Customers", color=color)
bars = ax1.bar(
    [r["NumOfProducts"] for r in prod_rows],
    [r["Total"] for r in prod_rows],
    color=color, alpha=0.6, width=0.5
)
ax1.tick_params(axis="y", labelcolor=color)

# Line chart: Churn Rate (%)
ax2 = ax1.twinx()
color = "#d95f02"
ax2.set_ylabel("Churn Rate (%)", color=color)
ax2.plot(
    [r["NumOfProducts"] for r in prod_rows],
    [r["ChurnRate"] for r in prod_rows],
    color=color, marker="o", linewidth=2.5
)
ax2.tick_params(axis="y", labelcolor=color)

plt.title("Customer Volume vs. Churn Rate by Number of Products")
plt.tight_layout()
plt.savefig(f"{GRAPHS_DIR}/churn_by_products.png")
plt.show()

In [0]:
spark.sql("""
    SELECT 
        Geography, 
        COUNT(*) AS TotalCustomers, 
        ROUND(SUM(Exited)/TotalCustomers*100, 1) AS PercentLost
    FROM churn
    GROUP BY Geography
    ORDER BY PercentLost DESC
""").show()

spark.sql("""
    SELECT 
        Gender,
        COUNT(*) AS TotalCustomers,
        ROUND(SUM(Exited)/TotalCustomers*100, 1) AS PercentLost
    FROM churn
    GROUP BY Gender
    ORDER BY PercentLost DESC;
""").show()

## ML branch
### preprocessing → stratified split → partitioned Delta → model input

The EDA branch above works on `some_churn` (stratified sample, unpartitioned — samples stay small by design).
The ML branch below works on the **full** dataset: only sampling bounds EDA compute — a churn model is only as good as the data it trains on.

1. Preprocess (drop IDs, encode categoricals, assemble ALL predictors into one `features` vector) and keep **only** `features`, the label `Exited`, and `Geography` — the raw columns are not needed downstream
2. Split train/test **stratified on y** (`Exited`): the top 80% of each class goes to train, so class balance is identical on both sides
3. Write once as Delta with `partitionBy("Geography", "is_train")`: **country first, then train/test**
4. Read back; model training consumes `train_df` / `test_df`

Every country appears in both the train and test directories: the split is the *second* partition level,
so no whole country ends up on one side of the split.

In [0]:
# ML branch: FULL dataset (not the EDA sample some_churn). Reuses churn_csv from the raw read.
# ID_COLS comes from the setup cell — no local redefinition.
ml_base = churn_csv.drop(*ID_COLS)

# Encode categoricals as numeric features. Geography plays both roles:
#   - kept as the string column that physically partitions the Delta table (directory key)
#   - encoded to Geography_idx so the model still gets country signal
gender_indexer = StringIndexer(inputCol="Gender", outputCol="Gender_idx").fit(ml_base)
geo_indexer = StringIndexer(inputCol="Geography", outputCol="Geography_idx").fit(ml_base)

ml_features = geo_indexer.transform(gender_indexer.transform(ml_base))

# Assemble every predictor into ONE features vector. Excluded: Exited (the label y),
# Geography (the partition column — the model gets its signal via Geography_idx),
# and the raw Gender string (superseded by Gender_idx).
FEATURE_EXCLUDE = {"Exited", "Geography", "Gender"}
feature_cols = [c for c in ml_features.columns if c not in FEATURE_EXCLUDE]

# Keep ONLY what the model needs: the vectorized features, the label y, and the
# partition key. The split and the partitioned write below operate on this compact
# frame — not on the wide dataset with all its raw columns.
model_input = (
    VectorAssembler(inputCols=feature_cols, outputCol="features")
    .transform(ml_features)
    .select("Geography", "Exited", "features")
)

print("feature_cols:", feature_cols)
model_input.printSchema()

In [0]:
# TRAIN_FRACTION / SPLIT_SEED come from the setup cell — one seed also drives the
# EDA sample, so both random steps are reproducible together.
# Stratified split on y (Exited): rank rows WITHIN each class and send the top
# TRAIN_FRACTION of each class to train, so the churn/retain proportions carry over
# exactly to both sides — important on imbalanced data like this.
stratified_split = (
    model_input
    # Materialize rand first so the window can order by it as a plain column
    .withColumn("__rand", F.rand(seed=SPLIT_SEED))
    .withColumn("class_rank", F.row_number().over(Window.partitionBy("Exited").orderBy("__rand")))
    .withColumn("class_size", F.count(F.lit(1)).over(Window.partitionBy("Exited")))
    .withColumn("is_train", F.col("class_rank") <= F.ceil(F.col("class_size") * TRAIN_FRACTION))
    .drop("class_rank", "class_size", "__rand")
)

# Partition FIRST by country, THEN by train/test. Directory layout:
#   PARTITIONED_DIR/Geography=France/is_train=true/...
#   PARTITIONED_DIR/Geography=France/is_train=false/...
# Every country appears on both sides — the split is the second partition level,
# so no whole country lands on one side of the train/test split.
# mode("overwrite") keeps notebook re-runs idempotent. overwriteSchema is needed
# here because this write also changes the table's schema/partitioning versus
# any earlier layout (e.g. Geography-only partitioning) already at that path.
(
    stratified_split.write
    .format("delta")
    .partitionBy("Geography", "is_train")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .save(PARTITIONED_DIR)
)
print(f"Wrote Delta table partitioned by Geography, then is_train, to {PARTITIONED_DIR}")

In [0]:
# Downstream ML reads from the partitioned table — never re-derives from the raw CSV.
ml_data = spark.read.format("delta").load(PARTITIONED_DIR)

# Sanity check: partition layout — one Geography=France/ ... directory per country
print("Partitions:", [r["Geography"] for r in ml_data.select("Geography").distinct().collect()])

# Train/test from the is_train partition level: both splits span all countries
train_df = ml_data.where(F.col("is_train"))
test_df = ml_data.where(~F.col("is_train"))

print("total:", ml_data.count(), "| train:", train_df.count(), "| test:", test_df.count())
display(ml_data.groupBy("Geography", "is_train").count().orderBy("Geography", "is_train"))

## Model training & experiment tracking

Four model families compete under **one MLflow experiment**. Every run is fitted once and evaluated on **both** train and test. The train/test gap is the overfitting signal — and each run gets a distinct auto-incrementing name (`run<N>_<family>_<date>`), so repeated sessions never overwrite or blur the experiment history.

1. Select/create the experiment; alias compact train/test frames carrying `features` + `Exited`
2. Standardize features with a **train-fitted** `RobustScaler`
3. Helpers: `f1_macro` and `train_test_log_model` (fit, evaluate AUC-ROC / macro F1 / accuracy on both splits, log params + metrics)
4. `train_models` loops over the candidate families, names each run, and returns fitted models + run IDs in `results`

In [0]:
# mlflow / MlflowClient are imported once in the setup cell.

# Create the experiment if it does not exist, and make it the default
# destination for every mlflow.start_run() in this notebook.
if mlflow.get_experiment_by_name(ML_EXPERIMENT) is None:
    mlflow.create_experiment(ML_EXPERIMENT)
EXPERIMENT = mlflow.set_experiment(ML_EXPERIMENT)
EXPERIMENT_ID = EXPERIMENT.experiment_id
print(f"Logging runs to experiment '{ML_EXPERIMENT}' (id: {EXPERIMENT_ID})")

In [0]:
# Spark ML consumes ONE DataFrame holding BOTH the features vector and the label —
# an sklearn-style X/y split (separate frames) cannot be fed to .fit(). The
# partitioned Delta table already carries exactly those columns, so alias compact
# frames for training/evaluation.
# .cache() is deliberate: LogisticRegression reads its input repeatedly
# (fit + summary + transform), and without it every access re-scans the Delta files.
train_ml = train_df.select("features", "Exited").cache()
test_ml = test_df.select("features", "Exited").cache()

train_rows = train_ml.count()
test_rows = test_ml.count()
train_churn = train_ml.select(F.mean("Exited")).first()[0]
test_churn = test_ml.select(F.mean("Exited")).first()[0]

print(f"train rows: {train_rows} | test rows: {test_rows}")
print(f"churn rate train: {train_churn:.3f} | test: {test_churn:.3f} (stratified split keeps them equal)")

In [0]:
# The scaler is fit on TRAIN ONLY, then applied to both sides — fitting it on the
# full dataset would let test-set medians/IQRs leak into training.
scaler = RobustScaler(inputCol="features", outputCol="scaled_features")
scaler_model = scaler.fit(train_ml)
scaled_train = scaler_model.transform(train_ml).cache()
scaled_test = scaler_model.transform(test_ml).cache()  # transformed with train-fitted statistics

def f1_macro(preds, label_col="Exited"):
    """Macro-averaged F1: the unweighted mean of the per-class F1 scores."""
    cm = (preds.groupBy(F.col(label_col).alias("label"), F.col("prediction").alias("pred"))
               .count()
               .collect())
    counts = {(float(r["label"]), float(r["pred"])): r["count"] for r in cm}

    def _f1(c):
        tp = counts.get((float(c),     float(c)),     0)
        fp = counts.get((float(1 - c),  float(c)),     0)  # predicted c, true is other
        fn = counts.get((float(c),     float(1 - c)), 0)  # true c, predicted other
        return 2 * tp / (2 * tp + fp + fn) if tp > 0 else 0.0

    return (_f1(0) + _f1(1)) / 2.0

In [0]:
def train_test_log_model(
    estimator,
    train_df,
    run_name: str,
    model_family: str,
    test_df=None,
    label_col: str = "Exited"
):
    """
    Fits a PySpark ML estimator on train_df and evaluates AUC-ROC, F1-macro and
    accuracy on train_df and, if provided, test_df. Everything is logged to MLflow.
    """
    auc_eval = BinaryClassificationEvaluator(
        labelCol=label_col, rawPredictionCol="rawPrediction", metricName="areaUnderROC"
    )
    acc_eval = MulticlassClassificationEvaluator(
        labelCol=label_col, predictionCol="prediction", metricName="accuracy"
    )

    splits = {"train": train_df}
    if test_df is not None:
        splits["test"] = test_df

    with mlflow.start_run(run_name=run_name) as run:
        fitted_model = estimator.fit(train_df)
        mlflow.set_tag("model_family", model_family)

        metrics = {}
        for split, df in splits.items():
            preds = fitted_model.transform(df)
            scores = {
                "auc_roc": auc_eval.evaluate(preds),
                "f1_macro": f1_macro(preds),
                "accuracy": acc_eval.evaluate(preds),
            }
            for name, value in scores.items():
                metrics[f"{split}_{name}"] = value
                print(f"[{run_name}] {split.capitalize()} {name}: {value:.4f}")

        # Feature count: tree models expose numFeatures, pipelines may not
        n_features = getattr(fitted_model, "numFeatures", None)
        if n_features is None:
            first = train_df.select(estimator.getFeaturesCol()).first()
            n_features = len(first[0]) if first else None

        params = {"n_features": n_features}
        params.update({f"{split}_rows": df.count() for split, df in splits.items()})

        mlflow.log_params(params)
        mlflow.log_metrics(metrics)
        run_id = run.info.run_id

    print(f"MLflow run recorded: {run_id}\n")
    return fitted_model, run_id

In [0]:
def train_models(train_df, test_df, models):
    """
    Train and log each estimator in `models` to an MLflow experiment.

    Parameters
    ----------
    train_df : DataFrame
    test_df : DataFrame
    models : dict[str, estimator]
        Maps model family name to an unfitted estimator.

    Returns
    -------
    dict[str, dict]
        {family_name: {"model": fitted_model, "run_id": run_id}}
    """
    client = MlflowClient()
    print(f"Logging runs to experiment '{EXPERIMENT}' (id: {EXPERIMENT_ID})")

    # Count existing runs per model family from MLflow tags
    run_counts = {}
    for run in client.search_runs(experiment_ids=[EXPERIMENT_ID], max_results=50000):
        family = run.data.tags.get("model_family")
        if family is not None:
            run_counts[family] = run_counts.get(family, 0) + 1

    today = date.today()
    results = {}

    for name, model in models.items():
        nr = run_counts.get(name, 0) + 1
        run_name = f"run{nr}_{name}_{today}"

        fitted_model, run_id = train_test_log_model(
            estimator=model,
            train_df=train_df,
            test_df=test_df,
            run_name=run_name,
            model_family=name,
        )

        run_counts[name] = nr
        results[name] = {"model": fitted_model, "run_id": run_id}

    return results

### Model definitions & training runs

Four candidates, each on the feature view that suits it:

* **Logistic Regression** on `scaled_features` (scale-sensitive) with elastic-net regularization
* **Random Forest** and **Gradient Boosting** on the raw `features` vector — tree models are scale-invariant
* **Naive Bayes** (multinomial) on the raw non-negative `features` — a deliberately simple baseline; it expects count-like features, so weak scores are expected

`MODELS` maps each family name to its estimator (the names double as the `model_family` tags in MLflow); `train_models` fits them all and stores fitted models + run IDs in `results`.

In [0]:
# One estimator per candidate family. LABEL comes from the setup cell; the dict
# keys are the model_family tags used to name runs in the MLflow experiment.
lr = LogisticRegression(featuresCol="scaled_features", labelCol=LABEL, elasticNetParam=0.1)
rf = RandomForestClassifier(featuresCol="features", labelCol=LABEL, numTrees=100, maxDepth=4)
nb = NaiveBayes(featuresCol="features", labelCol=LABEL, modelType="multinomial")
gb = GBTClassifier(featuresCol="features", labelCol=LABEL, maxIter=100, maxDepth=3)

MODELS = {
    "Naive Bayes": nb,
    "Logistic Regression": lr,
    "Random Forest": rf,
    "Gradient Boosting": gb,
}

In [0]:
# MODELS is defined in the cell above; results holds fitted models + MLflow run IDs
results = train_models(
    train_df=scaled_train,
    test_df=scaled_test,
    models=MODELS,
)

### Final model selection

Every run recorded in the experiment is compared on **test** AUC-ROC — test metrics, not train, because the model that memorizes the training set best is not the one that generalizes. The best run per family is reported; the overall winner is locked in as the final model.

With these features the winner is **Gradient Boosting** (`maxIter=100`, `maxDepth=3`): best test AUC-ROC (~0.85) and macro F1 among the four families, with only a small train/test gap.

In [0]:
# --- Final model selection ---
# Compare every logged run in the experiment on TEST AUC-ROC, take the best run
# per model family, and select the overall winner as the final model.
client = MlflowClient()

best_per_family = {}
for run in client.search_runs(experiment_ids=[EXPERIMENT_ID], max_results=50000):
    family = run.data.tags.get("model_family")
    auc = run.data.metrics.get("test_auc_roc")
    if family is None or auc is None:
        continue
    if family not in best_per_family or auc > best_per_family[family]["test_auc_roc"]:
        best_per_family[family] = {
            "test_auc_roc": auc,
            "test_f1_macro": run.data.metrics.get("test_f1_macro", float("nan")),
            "test_accuracy": run.data.metrics.get("test_accuracy", float("nan")),
            "run_name": run.info.run_name,
            "run_id": run.info.run_id,
        }

print("Best run per model family (by test AUC-ROC):")
for family, m in sorted(best_per_family.items(), key=lambda kv: -kv[1]["test_auc_roc"]):
    print(
        f"  {family:<20} auc_roc={m['test_auc_roc']:.4f}  "
        f"f1_macro={m['test_f1_macro']:.4f}  "
        f"accuracy={m['test_accuracy']:.4f}  ({m['run_name']})"
    )

# Overall winner = the family with the best test AUC-ROC. The fitted model is
# already in `results` from the training cell, so no re-fit is needed.
final_name = max(best_per_family, key=lambda f: best_per_family[f]["test_auc_roc"])
final_model = results[final_name]["model"]
final_run_id = best_per_family[final_name]["run_id"]
print(f"\nSelected final model: {final_name} (MLflow run {final_run_id})")